# Final Project — Beyond the Buffer (Sparse-Graph Fixed)
## Door-to-Door Food Retail Accessibility, Residential Demand, and Network Resilience in Upper Manhattan

**Yizhang Mu**  
**Mapping Systems — Final Project**  
**Submission version: July 29, 2026**

This project expands the course exercise on **networks and distance**. The tutorial calculates routes from a single point to restaurants and introduces H3 cells as an abstract network. This final project changes both the scale and the analytical question. It:

- allocates residential demand from MapPLUTO tax lots to H3 cells using a **proportional spatial split** rather than assigning every lot to one centroid;
- measures **door-to-door walking distance**, including the connection from each origin and store to the pedestrian graph;
- compares circular Euclidean buffers with network-based accessibility;
- evaluates how priority rankings change under different weighting assumptions; and
- stress-tests the system through multiple large-store closure scenarios.

The project does **not** claim that distance alone defines food security. It asks where conventional proximity maps simplify access, where those simplifications matter, and which locations deserve field verification.

## Project diagram

![Project workflow diagram](project_diagram.png)

## Short description of goals

Food-access maps often draw a circle around each store and assume that every location inside the circle has equal access. That representation ignores street connectivity, parks, campuses, superblocks, limited crossings, slopes, entrances, and other barriers. It can also treat every residential location as equally important and every store as equivalent.

This project tests those assumptions across Manhattan Community Districts 9–12. Residential units recorded in MapPLUTO are proportionally allocated to H3 demand cells. For each cell, the analysis compares straight-line and corrected door-to-door network distance to the nearest licensed retail food store, counts stores reachable within an estimated ten-minute walk, measures circular-buffer overcount, reports known store floor area separately from store counts, evaluates ranking sensitivity, and simulates the closure of three large stores with known floor area.

The final outputs are a reproducible notebook, a project diagram, saved figures, diagnostic tables, GeoJSON files, and an interactive MapLibre map.

## Research questions

1. **Where does Euclidean proximity overstate door-to-door pedestrian access to retail food stores?**
2. **Which residential-demand cells have the longest corrected network distance and the fewest stores within a ten-minute walk?**
3. **How much does centroid allocation differ from proportional allocation of residential units?**
4. **How stable are priority locations when the analytical weights change?**
5. **Which residential areas are most affected by the modeled closure of major stores with known floor area?**
6. **How can this method become a foundation for later research on food logistics, campus boundaries, delivery labor, affordability, and food quality?**

## How this expands the tutorials

The project is anchored in the **Networks** tutorial but deliberately extends it by combining methods from the full tutorial sequence:

| Tutorial method | Final-project extension |
|---|---|
| One origin point | Hundreds of H3 residential-demand origins |
| Tax-lot attributes | Proportional allocation from residential lots to H3 cells |
| Restaurants in one NTA | Licensed retail food stores across and around four community districts |
| One shortest route or one cuisine | Multi-source nearest-store analysis and ten-minute accessibility |
| Node-to-node network distance | Door-to-door distance including origin and destination snap distances |
| Euclidean vs. network distance | Detour ratio, circular-buffer overcount, and disagreement diagnostics |
| One analytical interpretation | Three priority-weight scenarios and ranking-stability tests |
| Existing network only | Three store-closure resilience scenarios |
| Static notebook result | Saved figures, GeoJSON exports, and a MapLibre interface |

The workflow also follows the tutorials' broader habits: inspect raw data before transformation, verify fields, align coordinate reference systems, cache API responses, expose modeling assumptions, visualize uncertainty, and reflect on bias rather than treating one map as neutral.

## Data and source notes

| Dataset | Role in the project | Important limitation |
|---|---|---|
| **NYC Department of City Planning MapPLUTO** | Tax-lot geometry and `UnitsRes` residential-unit proxy | Housing units are not population, household size, income, or food insecurity |
| **New York State Retail Food Stores** | Licensed retail food establishments and partial square-footage records | Licensing does not confirm current opening status, affordability, quality, or complete floor area |
| **OpenStreetMap pedestrian network** | Walkable graph used for shortest paths | Coverage may omit gates, stairs, informal paths, temporary restrictions, or crossing delay |

**Official source endpoints**

- MapPLUTO FeatureServer: `https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/MAPPLUTO/FeatureServer/0`
- Retail Food Stores dataset: `https://data.ny.gov/resource/9a8c-vfzj.geojson`
- OpenStreetMap network requested through OSMnx

The first successful run records retrieval time and caches the source files locally. The notebook reports missing values, excluded records, network-snap distances, allocation residuals, and store-size coverage before interpreting the results.

## 1. Import libraries

The import structure follows the course notebooks: tabular data, spatial data, plotting, requests, networks, geometry, and helper functions are loaded at the beginning. The first run requires internet access; successful requests are cached in the local `data` directory.

In [ ]:
# Install missing packages from the project root if needed:
# %pip install -r requirements.txt

In [ ]:
from __future__ import annotations

from collections import defaultdict
from datetime import datetime, timezone
from itertools import count
from pathlib import Path
import heapq
import json
import math
import os
import re
import warnings

import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from IPython.display import Markdown, display
from shapely.geometry import LineString, Point, Polygon, mapping, shape
from shapely import wkt
from tqdm.auto import tqdm

try:
    from cdptools import utils
    utils.set_axis_off()
except ImportError:
    print("cdptools is not installed; the notebook will use standard matplotlib axes.")

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

## 2. Project parameters and transparent assumptions

Important analytical choices are kept together so they can be inspected and revised. Distances are calculated in a meter-based UTM coordinate system. A ten-minute walk is represented as 800 meters, following the tutorial's simplified assumption of 80 meters per minute.

Three priority-weight scenarios are used instead of presenting one set of weights as objectively correct. The final robust priority score is the mean of the three scenarios, while rank spread reveals cells whose classification is especially sensitive to the chosen values.

In [ ]:
def locate_project_root() -> Path:
    """Locate the project folder regardless of how VSCode launches the kernel."""
    notebook_names = {"Final_Project_Beyond_the_Buffer_FINAL_FIXED.ipynb", "Final_Project_Beyond_the_Buffer_100.ipynb"}
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if any((candidate / name).exists() for name in notebook_names) and (candidate / "web").exists():
            return candidate.resolve()
        for nested_name in ["Beyond_the_Buffer_VSCode_Final_Fixed", "final_project_beyond_buffer_100"]:
            nested = candidate / nested_name
            if any((nested / name).exists() for name in notebook_names) and (nested / "web").exists():
                return nested.resolve()
    # Safe fallback when the notebook is renamed but remains in the project folder.
    return Path.cwd().resolve()


PROJECT_ROOT = locate_project_root()
DATA_DIR = PROJECT_ROOT / "data"
WEB_DATA_DIR = PROJECT_ROOT / "web" / "data"
FIGURE_DIR = PROJECT_ROOT / "figures"
TABLE_DIR = PROJECT_ROOT / "tables"

for directory in [DATA_DIR, WEB_DATA_DIR, FIGURE_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

MAPPLUTO_QUERY_URL = (
    "https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/"
    "MAPPLUTO/FeatureServer/0/query"
)
RETAIL_STORES_URL = "https://data.ny.gov/resource/9a8c-vfzj.json"

COMMUNITY_DISTRICTS = [109, 110, 111, 112]
COMMUNITY_DISTRICT_NAMES = {
    109: "Morningside Heights / Hamilton Heights",
    110: "Central Harlem",
    111: "East Harlem",
    112: "Washington Heights / Inwood",
}

ANALYSIS_CRS = "EPSG:32618"  # UTM zone 18N; meter-based
H3_RESOLUTION = 9
WALKING_SPEED_M_PER_MIN = 80
ACCESS_MINUTES = 10
ACCESS_CUTOFF_M = WALKING_SPEED_M_PER_MIN * ACCESS_MINUTES
NETWORK_BUFFER_M = 1000
MAX_SNAP_DISTANCE_M = 200
GRAVITY_DECAY_M = 800
TOP_PRIORITY_COUNT = 15
CLOSURE_SCENARIO_COUNT = 3

PRIORITY_WEIGHT_SCENARIOS = {
    "balanced": {
        "demand": 0.30,
        "distance": 0.30,
        "scarcity": 0.25,
        "overcount": 0.15,
    },
    "demand_focused": {
        "demand": 0.50,
        "distance": 0.20,
        "scarcity": 0.20,
        "overcount": 0.10,
    },
    "access_focused": {
        "demand": 0.15,
        "distance": 0.40,
        "scarcity": 0.35,
        "overcount": 0.10,
    },
}

PLUTO_CACHE = DATA_DIR / "upper_manhattan_mappluto.geojson"
STORES_CACHE = DATA_DIR / "manhattan_retail_food_stores.geojson"
GRAPH_CACHE = DATA_DIR / "upper_manhattan_walk.graphml"
RUN_METADATA_PATH = DATA_DIR / "run_metadata.json"

RUN_METADATA = {
    "project": "Beyond the Buffer",
    "run_started_utc": datetime.now(timezone.utc).isoformat(),
    "community_districts": COMMUNITY_DISTRICTS,
    "h3_resolution": H3_RESOLUTION,
    "walking_speed_m_per_min": WALKING_SPEED_M_PER_MIN,
    "access_cutoff_m": ACCESS_CUTOFF_M,
    "analysis_crs": ANALYSIS_CRS,
}

## 3. Helper functions

The tutorials repeatedly turn a sequence of operations into reusable functions. The functions below handle ArcGIS pagination, Socrata requests, H3 geometry creation, plotting, and data validation. They also write local caches so the notebook does not repeatedly download the same data.

In [ ]:
def fetch_arcgis_geojson(
    query_url: str,
    where: str,
    out_fields: list[str],
    cache_path: Path,
    out_sr: int = 4326,
    page_size: int = 2000,
) -> gpd.GeoDataFrame:
    """Download all matching ArcGIS features and cache them as GeoJSON."""
    if cache_path.exists():
        print(f"Loading cached ArcGIS data: {cache_path}")
        return gpd.read_file(cache_path)

    all_features: list[dict] = []
    offset = 0

    while True:
        params = {
            "where": where,
            "outFields": ",".join(out_fields),
            "returnGeometry": "true",
            "outSR": out_sr,
            "resultOffset": offset,
            "resultRecordCount": page_size,
            "f": "geojson",
        }
        response = requests.get(query_url, params=params, timeout=120)
        response.raise_for_status()
        payload = response.json()

        if "error" in payload:
            raise RuntimeError(payload["error"])

        features = payload.get("features", [])
        all_features.extend(features)
        print(f"Downloaded {len(all_features):,} ArcGIS features")

        if len(features) < page_size:
            break
        offset += page_size

    gdf = gpd.GeoDataFrame.from_features(all_features, crs=f"EPSG:{out_sr}")
    gdf.to_file(cache_path, driver="GeoJSON")
    return gdf


def _point_from_socrata(value):
    """Convert Socrata point values (dict or WKT text) to a Shapely Point."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, dict):
        try:
            geom = shape(value)
            return geom if geom.geom_type == "Point" else None
        except Exception:
            return None
    if isinstance(value, str) and value.strip():
        try:
            geom = wkt.loads(value)
            return geom if geom.geom_type == "Point" else None
        except Exception:
            return None
    return None


def fetch_socrata_geojson(
    endpoint: str,
    cache_path: Path,
    county: str = "NEW YORK",
    page_size: int = 50000,
) -> gpd.GeoDataFrame:
    """
    Download Socrata records, filter county locally, construct point geometry,
    and cache the result. Local filtering avoids failures caused by changes in
    county capitalization in the public API.
    """
    if cache_path.exists():
        cached = gpd.read_file(cache_path)
        if not cached.empty:
            print(f"Loading cached Socrata data: {cache_path}")
            return cached
        cache_path.unlink(missing_ok=True)

    json_endpoint = re.sub(r"\.(geojson|json)$", ".json", endpoint)
    records: list[dict] = []
    offset = 0

    while True:
        params = {"$limit": page_size, "$offset": offset}
        response = requests.get(json_endpoint, params=params, timeout=120)
        response.raise_for_status()
        page = response.json()
        if not isinstance(page, list):
            raise RuntimeError(f"Unexpected Socrata response: {page}")
        records.extend(page)
        print(f"Downloaded {len(records):,} Socrata records")
        if len(page) < page_size:
            break
        offset += page_size

    if not records:
        raise ValueError("The Socrata dataset returned no records.")

    frame = pd.DataFrame(records)
    frame.columns = frame.columns.str.lower()
    require_columns(frame, ["county", "georeference"], "Retail Food Stores API")

    county_key = str(county).strip().casefold()
    frame = frame[
        frame["county"].fillna("").astype(str).str.strip().str.casefold().eq(county_key)
    ].copy()
    if frame.empty:
        available = sorted(
            pd.DataFrame(records).get("county", pd.Series(dtype=str))
            .dropna().astype(str).str.strip().unique().tolist()
        )
        raise ValueError(
            f"No records matched county={county!r}. Available county examples: {available[:20]}"
        )

    frame["geometry"] = frame["georeference"].map(_point_from_socrata)
    missing_geometry = int(frame["geometry"].isna().sum())
    frame = frame[frame["geometry"].notna()].copy()
    if frame.empty:
        raise ValueError(
            "Manhattan records were returned, but none contained usable point geometry."
        )

    gdf = gpd.GeoDataFrame(frame.drop(columns=["georeference"]), geometry="geometry", crs="EPSG:4326")
    gdf.to_file(cache_path, driver="GeoJSON")
    print(
        f"Retained {len(gdf):,} Manhattan point records; "
        f"dropped {missing_geometry:,} records without usable geometry."
    )
    return gdf


def h3_cell_polygon(cell_id: str) -> Polygon:
    """Convert an H3 cell id into a Shapely polygon in longitude/latitude order."""
    return Polygon([(lon, lat) for lat, lon in h3.cell_to_boundary(cell_id)])


def h3_cells_for_geometry(geometry, resolution: int) -> set[str]:
    """Return H3 cells covering a Polygon or MultiPolygon across h3-py versions."""
    polygons = list(geometry.geoms) if geometry.geom_type == "MultiPolygon" else [geometry]
    cells: set[str] = set()
    for polygon in polygons:
        if hasattr(h3, "geo_to_cells"):
            cells.update(h3.geo_to_cells(mapping(polygon), resolution))
        else:
            outer = [(lat, lon) for lon, lat in polygon.exterior.coords]
            holes = [
                [(lat, lon) for lon, lat in ring.coords]
                for ring in polygon.interiors
            ]
            latlng_polygon = h3.LatLngPoly(outer, *holes)
            cells.update(h3.polygon_to_cells(latlng_polygon, resolution))

    # Standard H3 polygon filling is center-based. Add one neighboring ring so
    # boundary-crossing tax lots are not lost before the exact overlay clips them.
    expanded_cells = set(cells)
    for cell_id in list(cells):
        expanded_cells.update(h3.grid_disk(cell_id, 1))
    return expanded_cells


def require_columns(frame: pd.DataFrame, columns: list[str], label: str) -> None:
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise KeyError(f"{label} is missing required fields: {missing}")


def classify_store_size(square_feet: float) -> str:
    if pd.isna(square_feet) or square_feet <= 0:
        return "Unknown"
    if square_feet < 2500:
        return "Small (<2,500 sq ft)"
    if square_feet < 10000:
        return "Medium (2,500–9,999 sq ft)"
    return "Large (10,000+ sq ft)"


def save_current_figure(filename: str, dpi: int = 220) -> Path:
    """Save the current Matplotlib figure to the project figures directory."""
    path = FIGURE_DIR / filename
    plt.gcf().savefig(path, dpi=dpi, bbox_inches="tight")
    return path


def plot_metric(
    gdf: gpd.GeoDataFrame,
    column: str,
    title: str,
    legend_label: str,
    cmap: str = "viridis",
    scheme: str | None = "quantiles",
    figsize: tuple[int, int] = (10, 12),
    filename: str | None = None,
):
    """Create a consistent choropleth with a safe classification fallback."""
    values = pd.to_numeric(gdf[column], errors="coerce")
    plot_gdf = gdf.copy()
    plot_gdf[column] = values

    kwargs = {
        "column": column,
        "legend": True,
        "cmap": cmap,
        "figsize": figsize,
        "edgecolor": "white",
        "linewidth": 0.15,
        "missing_kwds": {"color": "#eeeeee", "label": "Missing"},
    }

    use_scheme = scheme is not None and values.nunique(dropna=True) > 1
    if use_scheme:
        kwargs["scheme"] = scheme
        kwargs["k"] = min(5, int(values.nunique(dropna=True)))
        kwargs["legend_kwds"] = {"loc": "lower left", "title": legend_label}
    else:
        kwargs["legend_kwds"] = {
            "label": legend_label,
            "orientation": "horizontal",
            "shrink": 0.7,
        }

    try:
        ax = plot_gdf.plot(**kwargs)
    except (TypeError, ImportError, ValueError) as exc:
        # Different GeoPandas/mapclassify versions accept slightly different
        # legend arguments. Falling back to a continuous map keeps Run All alive.
        print(f"Classification fallback for {column}: {exc}")
        kwargs.pop("scheme", None)
        kwargs.pop("k", None)
        kwargs["legend_kwds"] = {
            "label": legend_label,
            "orientation": "horizontal",
            "shrink": 0.7,
        }
        ax = plot_gdf.plot(**kwargs)

    ax.set_title(title, pad=14)
    ax.set_axis_off()
    if filename is None:
        filename = re.sub(r"[^a-z0-9]+", "_", column.lower()).strip("_") + ".png"
    ax.figure.savefig(FIGURE_DIR / filename, dpi=220, bbox_inches="tight")
    plt.show()
    return ax


def edge_length(graph: nx.Graph, u, v) -> float:
    """Return the shortest parallel edge length between adjacent graph nodes."""
    data = graph.get_edge_data(u, v)
    if data is None:
        raise KeyError(f"No edge between {u!r} and {v!r}")
    if graph.is_multigraph():
        return min(float(attrs.get("length", 1.0)) for attrs in data.values())
    return float(data.get("length", 1.0))


def multi_source_store_dijkstra(
    graph: nx.Graph,
    stores_frame: pd.DataFrame,
) -> tuple[dict, dict, dict]:
    """
    Compute nearest-store distance from every network node.

    Each store begins with its own store-to-node snap distance. The returned
    distance therefore measures network-node-to-store-door cost, while the
    origin snap distance is added later for a complete door-to-door measure.
    """
    required = ["network_node", "snap_distance_m", "license_number"]
    require_columns(stores_frame, required, "Store source table")

    ordered = stores_frame.copy()
    ordered["_known_sqft"] = ordered.get("square_footage", pd.Series(index=ordered.index)).fillna(-1)
    ordered = ordered.sort_values(
        ["snap_distance_m", "_known_sqft"], ascending=[True, False]
    )

    best_distance: dict = {}
    nearest_store: dict = {}
    toward_store: dict = {}
    queue: list[tuple[float, int, object, object]] = []
    serial = count()

    for row in ordered.itertuples():
        node = row.network_node
        offset = float(row.snap_distance_m)
        if offset < best_distance.get(node, math.inf):
            best_distance[node] = offset
            nearest_store[node] = row.license_number
            toward_store[node] = None
            heapq.heappush(queue, (offset, next(serial), node, row.license_number))

    while queue:
        distance, _, node, store_license = heapq.heappop(queue)
        if distance > best_distance.get(node, math.inf) + 1e-9:
            continue

        for neighbor in graph.neighbors(node):
            candidate = distance + edge_length(graph, node, neighbor)
            if candidate + 1e-9 < best_distance.get(neighbor, math.inf):
                best_distance[neighbor] = candidate
                nearest_store[neighbor] = store_license
                toward_store[neighbor] = node
                heapq.heappush(
                    queue, (candidate, next(serial), neighbor, store_license)
                )

    return best_distance, nearest_store, toward_store


def reconstruct_node_path(origin_node, toward_store: dict, max_steps: int = 10000) -> list:
    """Reconstruct an origin-to-store-node path from the Dijkstra pointer map."""
    path = [origin_node]
    current = origin_node
    seen = {current}
    for _ in range(max_steps):
        next_node = toward_store.get(current)
        if next_node is None:
            break
        if next_node in seen:
            raise RuntimeError("Cycle detected while reconstructing route.")
        path.append(next_node)
        seen.add(next_node)
        current = next_node
    return path


def weighted_average(values: pd.Series, weights: pd.Series) -> float:
    valid = values.notna() & weights.notna() & weights.gt(0)
    if not valid.any():
        return np.nan
    return float(np.average(values[valid], weights=weights[valid]))


def weighted_share(mask: pd.Series, weights: pd.Series) -> float:
    valid = mask.notna() & weights.notna() & weights.gt(0)
    if not valid.any():
        return np.nan
    return float(weights[valid & mask.astype(bool)].sum() / weights[valid].sum())


def percentile_high(values: pd.Series) -> pd.Series:
    """Score larger values as higher priority on a 0–1 percentile scale."""
    return values.rank(method="average", pct=True)


def percentile_low(values: pd.Series) -> pd.Series:
    """Score smaller values as higher priority on a 0–1 percentile scale."""
    return values.rank(method="average", pct=True, ascending=False)

## 4. Download and inspect MapPLUTO

The ArcGIS request filters before import, following the proportional-split tutorial. This reduces memory use and keeps only the four community districts that define the study area.

In [ ]:
pluto_fields = [
    "OBJECTID",
    "BBL",
    "CD",
    "Address",
    "ZipCode",
    "LandUse",
    "OwnerName",
    "LotArea",
    "BldgArea",
    "ResArea",
    "RetailArea",
    "UnitsRes",
    "UnitsTotal",
    "NumBldgs",
    "NumFloors",
    "BuiltFAR",
    "ResidFAR",
    "Latitude",
    "Longitude",
]

pluto_where = f"CD IN ({','.join(map(str, COMMUNITY_DISTRICTS))})"
pluto_raw = fetch_arcgis_geojson(
    MAPPLUTO_QUERY_URL,
    where=pluto_where,
    out_fields=pluto_fields,
    cache_path=PLUTO_CACHE,
)

In [ ]:
print(f"MapPLUTO rows: {len(pluto_raw):,}")
print(f"MapPLUTO columns: {len(pluto_raw.columns):,}")
pluto_raw.head()

In [ ]:
pluto_raw.columns.tolist()

In [ ]:
ax = pluto_raw.plot(figsize=(9, 12), color="#d9d9d9", edgecolor="white", linewidth=0.1)
ax.set_title("MapPLUTO lots in Manhattan Community Districts 109–112")
ax.set_axis_off()
plt.show()

### Clean residential demand and validate geometry

`UnitsRes` is converted to numeric form and only positive residential lots are retained. Invalid polygons are repaired where possible before area-based allocation. The original unit total is stored so the proportional split can be checked for conservation.

In [ ]:
require_columns(pluto_raw, ["BBL", "CD", "UnitsRes", "geometry"], "MapPLUTO")

pluto = pluto_raw.copy()
for field in [
    "BBL", "CD", "LotArea", "BldgArea", "ResArea", "RetailArea",
    "UnitsRes", "UnitsTotal", "NumBldgs", "NumFloors", "BuiltFAR", "ResidFAR",
]:
    if field in pluto.columns:
        pluto[field] = pd.to_numeric(pluto[field], errors="coerce")

pluto = pluto[pluto.geometry.notna() & ~pluto.geometry.is_empty].copy()
pluto.geometry = pluto.geometry.make_valid()
residential_lots = pluto[pluto["UnitsRes"].fillna(0).gt(0)].copy()
SOURCE_RESIDENTIAL_UNITS = float(residential_lots["UnitsRes"].sum())

print(f"All tax lots: {len(pluto):,}")
print(f"Residential lots: {len(residential_lots):,}")
print(f"Recorded residential units: {SOURCE_RESIDENTIAL_UNITS:,.0f}")
print(f"Invalid geometries after repair: {(~residential_lots.geometry.is_valid).sum():,}")

In [ ]:
residential_lots["UnitsRes"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])

In [ ]:
residential_lots["UnitsRes"].clip(
    upper=residential_lots["UnitsRes"].quantile(0.99)
).hist(bins=35, figsize=(9, 5))
plt.title("Distribution of residential units per tax lot (clipped at 99th percentile)")
plt.xlabel("Residential units")
plt.ylabel("Tax lots")
plt.show()

## 5. Create the residential core and buffered network boundary

Residential origins are restricted to the dissolved extent of Community Districts 9–12. The pedestrian graph and store destinations use a one-kilometer buffer so residents near the study edge can reach stores immediately outside the four districts. This avoids an artificial edge effect while keeping the analytical origins clearly defined.

In [ ]:
pluto_m = pluto.to_crs(ANALYSIS_CRS)
study_core_m = gpd.GeoDataFrame(
    {"name": ["Upper Manhattan residential core"]},
    geometry=[pluto_m.geometry.union_all()],
    crs=ANALYSIS_CRS,
)
study_boundary_m = gpd.GeoDataFrame(
    {"name": ["Upper Manhattan network catchment"]},
    geometry=[study_core_m.geometry.iloc[0].buffer(NETWORK_BUFFER_M)],
    crs=ANALYSIS_CRS,
)
study_core = study_core_m.to_crs("EPSG:4326")
study_boundary = study_boundary_m.to_crs("EPSG:4326")

ax = study_boundary.boundary.plot(figsize=(8, 11), color="black", linewidth=1.2)
study_core.plot(ax=ax, color="#dddddd", edgecolor="white", linewidth=0.05)
ax.set_title("Residential core and buffered network catchment")
ax.set_axis_off()
save_current_figure("01_study_area.png")
plt.show()

## 6. Download and inspect licensed retail food stores

The statewide dataset is filtered to New York County before import. The point records are then clipped to the buffered study area. As in the tutorials, the raw values are inspected before any categorical filter is applied.

### API reliability note

The New York State dataset currently stores Manhattan county values as uppercase `NEW YORK`. To avoid a brittle server-side text filter, the helper downloads the statewide table, normalizes county capitalization locally, and converts the `georeference` field into point geometry. If an older empty cache exists, delete `data/manhattan_retail_food_stores.geojson` and rerun this section.


In [ ]:
stores_raw = fetch_socrata_geojson(
    RETAIL_STORES_URL,
    cache_path=STORES_CACHE,
    county="NEW YORK",
)


In [ ]:
print(f"Raw Manhattan retail-food records: {len(stores_raw):,}")
print(f"Columns: {len(stores_raw.columns):,}")
stores_raw.head()

In [ ]:
stores_raw.columns.tolist()

In [ ]:
for field in ["operation_type", "estab_type"]:
    if field in stores_raw.columns:
        display(stores_raw[field].fillna("Missing").value_counts().head(25))

### Clean, classify, and audit store records

The notebook first normalizes column names and inspects `operation_type`. If a meaningful operation-type category containing the word “store” exists, that semantic field is used. Only if it is unavailable does the code fall back to the `A` establishment-code prefix used in the original draft. The chosen rule and record counts are printed so the filter is never hidden.

In [ ]:
stores = stores_raw.copy()
stores.columns = stores.columns.str.lower()
require_columns(stores, ["license_number", "geometry"], "Retail Food Stores")

if "square_footage" not in stores.columns:
    stores["square_footage"] = np.nan
stores["square_footage"] = pd.to_numeric(stores["square_footage"], errors="coerce")

for field in [
    "dba_name", "entity_name", "address_line_1", "address_line_2",
    "city", "zip_code", "operation_type", "estab_type",
]:
    if field not in stores.columns:
        stores[field] = pd.NA

stores["store_name"] = (
    stores["dba_name"].replace("", pd.NA)
    .fillna(stores["entity_name"])
    .fillna("Unnamed store")
)
stores["address"] = (
    stores["address_line_1"].fillna("").astype(str).str.strip()
    + " "
    + stores["address_line_2"].fillna("").astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

# The source dataset is already defined as licensed retail food stores; do not
# apply an undocumented establishment-code filter that could discard valid rows.
valid_geometry = stores.geometry.notna() & ~stores.geometry.is_empty
stores = stores[valid_geometry].copy()
STORE_FILTER_RULE = "all licensed retail-food records with usable point geometry"
stores.drop_duplicates(subset="license_number", keep="first", inplace=True)

stores = gpd.sjoin(
    stores,
    study_boundary[["geometry"]],
    how="inner",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")

stores["store_size"] = stores["square_footage"].map(classify_store_size)
stores["known_square_footage"] = stores["square_footage"].fillna(0).gt(0)

if stores.empty:
    raise ValueError(
        "No Manhattan retail-food points fall inside the study boundary. "
        "Delete data/manhattan_retail_food_stores.geojson and rerun the download cell."
    )

print(f"Store filtering rule: {STORE_FILTER_RULE}")
print(f"Stores inside buffered study boundary: {len(stores):,}")
print(f"Known square footage: {stores['known_square_footage'].mean():.1%}")


In [ ]:
stores["store_size"].value_counts(dropna=False)

In [ ]:
size_order = [
    "Small (<2,500 sq ft)",
    "Medium (2,500–9,999 sq ft)",
    "Large (10,000+ sq ft)",
    "Unknown",
]
stores["store_size"].value_counts().reindex(size_order).fillna(0).plot.bar(figsize=(9, 5))
plt.title("Licensed retail food stores by recorded floor-area class")
plt.xlabel("Store size class")
plt.ylabel("Store records")
plt.xticks(rotation=20, ha="right")
plt.show()

In [ ]:
ax = pluto.plot(figsize=(10, 13), color="#eeeeee", edgecolor="white", linewidth=0.05)
stores.plot(ax=ax, column="store_size", categorical=True, legend=True, markersize=18, alpha=0.85)
ax.set_title("Store-coded retail food establishments in the study area")
ax.set_axis_off()
plt.show()

## 7. Allocate residential demand to H3 cells using proportional split

The original draft assigned every tax lot to the H3 cell containing one representative point. That is fast, but it can move all units from a large lot into one cell. This version follows the logic of the proportional-split tutorial:

1. generate an H3 grid over the residential core;
2. intersect residential tax lots with the grid in a projected CRS;
3. calculate the share of each lot inside each H3 cell; and
4. allocate `UnitsRes` according to that area share.

A centroid-based estimate is also retained as a sensitivity comparison. The allocated total is checked against the original MapPLUTO total before analysis continues.

In [ ]:
# Build a complete H3 grid over the unbuffered residential core.
core_geometry_wgs84 = study_core.geometry.iloc[0]
h3_ids = sorted(h3_cells_for_geometry(core_geometry_wgs84, H3_RESOLUTION))
h3_grid = gpd.GeoDataFrame(
    {"h3_id": h3_ids},
    geometry=[h3_cell_polygon(cell_id) for cell_id in h3_ids],
    crs="EPSG:4326",
)
h3_grid_m = h3_grid.to_crs(ANALYSIS_CRS)

# Repair and project residential lots before measuring area.
residential_lots_m = residential_lots.to_crs(ANALYSIS_CRS).copy()
residential_lots_m.geometry = residential_lots_m.geometry.make_valid()
residential_lots_m["source_lot_area_m2"] = residential_lots_m.geometry.area
residential_lots_m = residential_lots_m[
    residential_lots_m["source_lot_area_m2"].gt(0)
].copy()

# Intersect lots and H3 cells, then allocate units by intersection share.
allocation_pieces = gpd.overlay(
    residential_lots_m[
        ["BBL", "CD", "UnitsRes", "source_lot_area_m2", "geometry"]
    ],
    h3_grid_m[["h3_id", "geometry"]],
    how="intersection",
    keep_geom_type=False,
)
allocation_pieces = allocation_pieces[
    allocation_pieces.geometry.notna() & ~allocation_pieces.geometry.is_empty
].copy()
allocation_pieces["intersection_area_m2"] = allocation_pieces.geometry.area
allocation_pieces = allocation_pieces[allocation_pieces["intersection_area_m2"].gt(0)].copy()
allocation_pieces["allocation_share"] = (
    allocation_pieces["intersection_area_m2"]
    / allocation_pieces["source_lot_area_m2"]
)
allocation_pieces["allocated_units"] = (
    allocation_pieces["UnitsRes"] * allocation_pieces["allocation_share"]
)

# Main H3 demand table.
h3_summary = (
    allocation_pieces.groupby("h3_id")
    .agg(
        residential_units=("allocated_units", "sum"),
        residential_lots=("BBL", "nunique"),
        intersected_lot_pieces=("BBL", "size"),
    )
    .reset_index()
)

# Assign the community district receiving the largest allocated-unit share.
district_allocation = (
    allocation_pieces.groupby(["h3_id", "CD"], as_index=False)["allocated_units"].sum()
)
dominant_cd = (
    district_allocation.sort_values("allocated_units", ascending=False)
    .drop_duplicates("h3_id")
    .rename(columns={"CD": "dominant_cd", "allocated_units": "dominant_cd_units"})
    [["h3_id", "dominant_cd", "dominant_cd_units"]]
)

h3_demand = (
    h3_grid.merge(h3_summary, on="h3_id", how="inner")
    .merge(dominant_cd, on="h3_id", how="left")
)
h3_demand = gpd.GeoDataFrame(h3_demand, geometry="geometry", crs="EPSG:4326")
h3_demand["dominant_cd"] = pd.to_numeric(h3_demand["dominant_cd"], errors="coerce").astype("Int64")
h3_demand["district_name"] = h3_demand["dominant_cd"].map(COMMUNITY_DISTRICT_NAMES)

# Centroid allocation retained only for sensitivity comparison.
residential_centroid_points = gpd.GeoDataFrame(
    residential_lots_m.drop(columns="geometry"),
    geometry=residential_lots_m.geometry.representative_point(),
    crs=ANALYSIS_CRS,
).to_crs("EPSG:4326")
residential_centroid_points["h3_id"] = residential_centroid_points.geometry.apply(
    lambda point: h3.latlng_to_cell(point.y, point.x, H3_RESOLUTION)
)
centroid_units = (
    residential_centroid_points.groupby("h3_id", as_index=False)["UnitsRes"]
    .sum()
    .rename(columns={"UnitsRes": "centroid_residential_units"})
)
h3_demand = h3_demand.merge(centroid_units, on="h3_id", how="left")
h3_demand["centroid_residential_units"] = h3_demand["centroid_residential_units"].fillna(0)
h3_demand["allocation_difference_units"] = (
    h3_demand["residential_units"] - h3_demand["centroid_residential_units"]
)
h3_demand["allocation_difference_pct"] = (
    h3_demand["allocation_difference_units"]
    / h3_demand["residential_units"].replace(0, np.nan)
    * 100
)

ALLOCATED_RESIDENTIAL_UNITS = float(h3_demand["residential_units"].sum())
ALLOCATION_RESIDUAL_UNITS = ALLOCATED_RESIDENTIAL_UNITS - SOURCE_RESIDENTIAL_UNITS
ALLOCATION_RESIDUAL_PCT = ALLOCATION_RESIDUAL_UNITS / SOURCE_RESIDENTIAL_UNITS * 100

print(f"H3 grid cells with residential demand: {len(h3_demand):,}")
print(f"Source residential units: {SOURCE_RESIDENTIAL_UNITS:,.2f}")
print(f"Proportionally allocated units: {ALLOCATED_RESIDENTIAL_UNITS:,.2f}")
print(
    f"Allocation residual: {ALLOCATION_RESIDUAL_UNITS:,.4f} units "
    f"({ALLOCATION_RESIDUAL_PCT:.6f}%)"
)

if abs(ALLOCATION_RESIDUAL_PCT) > 0.5:
    raise ValueError(
        "Residential allocation lost more than 0.5% of units. "
        "Inspect geometry validity and study-boundary coverage."
    )

RUN_METADATA["source_residential_units"] = SOURCE_RESIDENTIAL_UNITS
RUN_METADATA["allocated_residential_units"] = ALLOCATED_RESIDENTIAL_UNITS
RUN_METADATA["allocation_residual_pct"] = ALLOCATION_RESIDUAL_PCT

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 10))
h3_demand.plot(
    ax=axes[0],
    column="residential_units",
    cmap="viridis",
    scheme="quantiles",
    legend=True,
    edgecolor="white",
    linewidth=0.15,
)
axes[0].set_title("Proportionally allocated residential units")
axes[0].set_axis_off()

limit = h3_demand["allocation_difference_pct"].abs().quantile(0.95)
h3_demand.plot(
    ax=axes[1],
    column="allocation_difference_pct",
    cmap="coolwarm",
    vmin=-limit,
    vmax=limit,
    legend=True,
    edgecolor="white",
    linewidth=0.15,
)
axes[1].set_title("Centroid minus proportional allocation disagreement")
axes[1].set_axis_off()
plt.suptitle("Residential-demand allocation method sensitivity", y=0.94)
save_current_figure("04_residential_allocation_comparison.png")
plt.show()

allocation_diagnostics = pd.Series(
    {
        "Source residential units": SOURCE_RESIDENTIAL_UNITS,
        "Allocated residential units": ALLOCATED_RESIDENTIAL_UNITS,
        "Allocation residual (%)": ALLOCATION_RESIDUAL_PCT,
        "Median absolute cell difference (units)": h3_demand[
            "allocation_difference_units"
        ].abs().median(),
        "90th percentile absolute cell difference (units)": h3_demand[
            "allocation_difference_units"
        ].abs().quantile(0.90),
    }
)
allocation_diagnostics

## 8. Build or load the pedestrian network

OSMnx requests a pedestrian graph for the buffered study boundary. The largest connected component is retained by `retain_all=False`, matching the tutorial's emphasis on usable routes. The graph is projected to the analysis CRS and saved to GraphML so later runs can reuse it.

In [ ]:
if GRAPH_CACHE.exists():
    print(f"Loading cached graph: {GRAPH_CACHE}")
    G = ox.load_graphml(GRAPH_CACHE)
else:
    graph_polygon = study_boundary.geometry.iloc[0]
    G = ox.graph_from_polygon(
        graph_polygon,
        network_type="walk",
        simplify=True,
        retain_all=False,
    )
    G = ox.project_graph(G, to_crs=ANALYSIS_CRS)
    ox.save_graphml(G, GRAPH_CACHE)

# OSMnx 2.x returns a MultiDiGraph. Convert to an undirected graph for
# symmetric walking accessibility while preserving edge-length weights.
try:
    G_walk = ox.convert.to_undirected(G)
except AttributeError:
    G_walk = G.to_undirected()

network_nodes, network_edges = ox.graph_to_gdfs(G)
if network_nodes.crs is None or str(network_nodes.crs) != ANALYSIS_CRS:
    raise ValueError(f"Walking graph CRS is {network_nodes.crs}; expected {ANALYSIS_CRS}.")
print(f"Network nodes: {len(network_nodes):,}")
print(f"Network edges: {len(network_edges):,}")


In [ ]:
ax = network_edges.plot(figsize=(10, 13), color="black", linewidth=0.15)
stores.to_crs(ANALYSIS_CRS).plot(ax=ax, color="orange", markersize=8, alpha=0.65)
ax.set_title("Pedestrian network and retail food stores")
ax.set_axis_off()
plt.show()

## 9. Snap origins and stores to the network, then audit exclusions

Nearest-node snapping is an approximation. This version measures the connection distance from every H3 representative point and store point to its network node. Records beyond the tolerance are excluded and counted. These snap distances are then included in all door-to-door network calculations rather than reported only as diagnostics.

In [ ]:
# Create one representative point inside each H3 demand cell.
# GeoDataFrame already names the active geometry column "geometry", so do not
# call rename_geometry("geometry") afterwards.
h3_origin_points = gpd.GeoDataFrame(
    h3_demand.drop(columns="geometry").copy(),
    geometry=h3_demand.geometry.representative_point(),
    crs=h3_demand.crs,
).to_crs(ANALYSIS_CRS)

stores_m = stores.to_crs(ANALYSIS_CRS).copy()

# The graph, graph nodes, origins, and stores must all use the same projected CRS.
graph_crs = G.graph.get("crs")
if str(graph_crs).upper() != str(ANALYSIS_CRS).upper():
    print(f"Projecting graph from {graph_crs} to {ANALYSIS_CRS}")
    G = ox.project_graph(G, to_crs=ANALYSIS_CRS)
    network_nodes, network_edges = ox.graph_to_gdfs(G, nodes=True, edges=True)

assert h3_origin_points.crs == stores_m.crs == network_nodes.crs, (
    f"CRS mismatch: origins={h3_origin_points.crs}, stores={stores_m.crs}, "
    f"nodes={network_nodes.crs}"
)
assert not h3_origin_points.crs.is_geographic, "Snapping must use a projected CRS in meters."

h3_origin_points["network_node"] = ox.distance.nearest_nodes(
    G,
    X=h3_origin_points.geometry.x.to_numpy(),
    Y=h3_origin_points.geometry.y.to_numpy(),
)
stores_m["network_node"] = ox.distance.nearest_nodes(
    G,
    X=stores_m.geometry.x.to_numpy(),
    Y=stores_m.geometry.y.to_numpy(),
)

# Map graph-node coordinates once; this is faster and safer than repeated .loc calls.
node_geometry = network_nodes.geometry
h3_origin_points["snap_distance_m"] = [
    point.distance(node_geometry.loc[node])
    for point, node in zip(h3_origin_points.geometry, h3_origin_points["network_node"])
]
stores_m["snap_distance_m"] = [
    point.distance(node_geometry.loc[node])
    for point, node in zip(stores_m.geometry, stores_m["network_node"])
]

print("Graph CRS:", G.graph.get("crs"))
print("Origin CRS:", h3_origin_points.crs)
print("Store CRS:", stores_m.crs)
print("Origin snap-distance summary")
display(h3_origin_points["snap_distance_m"].describe())
print("Store snap-distance summary")
display(stores_m["snap_distance_m"].describe())


In [ ]:
origin_keep = h3_origin_points["snap_distance_m"].le(MAX_SNAP_DISTANCE_M)
store_keep = stores_m["snap_distance_m"].le(MAX_SNAP_DISTANCE_M)

excluded_origin_units = h3_origin_points.loc[~origin_keep, "residential_units"].sum()
excluded_store_count = int((~store_keep).sum())

h3_origin_points = h3_origin_points[origin_keep].copy()
stores_m = stores_m[store_keep].copy()

print(f"Origins retained after snapping: {len(h3_origin_points):,}")
print(f"Residential units excluded by snap tolerance: {excluded_origin_units:,.2f}")
print(f"Stores retained after snapping: {len(stores_m):,}")
print(f"Stores excluded by snap tolerance: {excluded_store_count:,}")

if stores_m.empty:
    raise ValueError("No store records remain after network snapping.")
if h3_origin_points.empty:
    raise ValueError("No residential origins remain after network snapping.")

RUN_METADATA["origin_cells_after_snap"] = int(len(h3_origin_points))
RUN_METADATA["residential_units_excluded_by_snap"] = float(excluded_origin_units)
RUN_METADATA["stores_after_snap"] = int(len(stores_m))
RUN_METADATA["stores_excluded_by_snap"] = excluded_store_count

## 10. Compare Euclidean and corrected door-to-door network distance

The Euclidean measure uses the H3 representative point and the actual store point. The network measure now includes three parts:

1. origin point → origin network node;
2. shortest path through the pedestrian graph; and
3. destination network node → actual store point.

A custom multi-source Dijkstra calculation initializes every store node with its own destination snap distance. The origin snap distance is added afterward. This fixes the original draft's node-to-node undercount and makes the detour ratio internally consistent.

In [ ]:
# Euclidean nearest-store join between actual origin and destination points.
store_join_fields = [
    "license_number", "store_name", "store_size", "square_footage", "geometry"
]
euclidean_join = gpd.sjoin_nearest(
    h3_origin_points[["h3_id", "geometry"]],
    stores_m[store_join_fields],
    how="left",
    distance_col="euclidean_m",
)
euclidean_join = (
    euclidean_join.sort_values(["h3_id", "euclidean_m"])
    .drop_duplicates("h3_id")
    .rename(
        columns={
            "license_number": "euclidean_store_license",
            "store_name": "euclidean_store_name",
        }
    )
)

In [ ]:
# Correct nearest-store network cost from every network node.
node_to_store_cost, nearest_store_license, toward_store = multi_source_store_dijkstra(
    G_walk,
    stores_m,
)
store_lookup = stores_m.drop_duplicates("license_number").set_index("license_number")

network_rows = []
for origin in h3_origin_points.itertuples():
    graph_node = origin.network_node
    license_number = nearest_store_license.get(graph_node, pd.NA)
    node_to_door_m = node_to_store_cost.get(graph_node, np.nan)
    total_door_to_door_m = (
        float(origin.snap_distance_m) + float(node_to_door_m)
        if pd.notna(node_to_door_m)
        else np.nan
    )
    store_record = (
        store_lookup.loc[license_number]
        if pd.notna(license_number) and license_number in store_lookup.index
        else None
    )
    network_rows.append(
        {
            "h3_id": origin.h3_id,
            "network_m": total_door_to_door_m,
            "network_graph_plus_store_snap_m": node_to_door_m,
            "network_store_license": license_number,
            "network_store_name": (
                store_record["store_name"] if store_record is not None else pd.NA
            ),
            "network_store_snap_m": (
                float(store_record["snap_distance_m"])
                if store_record is not None
                else np.nan
            ),
        }
    )

network_nearest = pd.DataFrame(network_rows)

In [ ]:
access = (
    h3_origin_points.drop(columns="geometry")
    .merge(
        euclidean_join[
            [
                "h3_id",
                "euclidean_m",
                "euclidean_store_license",
                "euclidean_store_name",
            ]
        ],
        on="h3_id",
        how="left",
    )
    .merge(network_nearest, on="h3_id", how="left")
)

access["walk_min_nearest"] = access["network_m"] / WALKING_SPEED_M_PER_MIN
access["detour_index"] = access["network_m"] / access["euclidean_m"].replace(0, np.nan)
access["nearest_store_changed"] = (
    access["euclidean_store_license"] != access["network_store_license"]
)
access["network_minus_euclidean_m"] = access["network_m"] - access["euclidean_m"]
access["distance_consistency_flag"] = access["network_minus_euclidean_m"].lt(-1.0)

consistency_violations = int(access["distance_consistency_flag"].sum())
print(f"Door-to-door network distances below Euclidean distance by >1 m: {consistency_violations}")
if consistency_violations:
    display(
        access.loc[
            access["distance_consistency_flag"],
            ["h3_id", "euclidean_m", "network_m", "network_minus_euclidean_m"],
        ].head(20)
    )

access[
    [
        "h3_id", "residential_units", "euclidean_m", "network_m",
        "walk_min_nearest", "detour_index", "nearest_store_changed",
        "snap_distance_m", "network_store_snap_m",
    ]
].describe(include="all")

## 11. Compare Euclidean and network distance

The diagonal line represents equal distance. Points above it require a longer route through the walking network. Cell size is not used as a visual weight here; residential demand is analyzed separately so the geometric comparison remains legible.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(access["euclidean_m"], access["network_m"], s=16, alpha=0.5)
maximum = np.nanmax([access["euclidean_m"].max(), access["network_m"].max()])
ax.plot([0, maximum], [0, maximum], linestyle="--", linewidth=1, color="black")
ax.set_xlabel("Euclidean distance to nearest store (m)")
ax.set_ylabel("Corrected door-to-door network distance (m)")
ax.set_title("Euclidean versus door-to-door walking distance")
save_current_figure("07_euclidean_vs_network_distance.png")
plt.show()

In [ ]:
access["detour_index"].clip(
    upper=access["detour_index"].quantile(0.99)
).hist(bins=35, figsize=(9, 5))
plt.axvline(1, linestyle="--", linewidth=1, color="black")
plt.xlabel("Detour index (door-to-door network / Euclidean distance)")
plt.ylabel("H3 demand cells")
plt.title("Distribution of corrected network detours")
save_current_figure("08_detour_index_distribution.png")
plt.show()

## 12. Count stores reachable within ten door-to-door minutes

This chapter uses a **chunked SciPy sparse-graph Dijkstra calculation** rather than running NetworkX once for every H3 origin. The graph is converted to a CSR matrix using the shortest parallel edge for each directed node pair. Unique origin nodes are processed in small batches, and only distances to store nodes are retained.

A store counts as reachable only when:

`origin snap + graph distance + store snap ≤ 800 meters`

The calculation prints progress and elapsed time, saves a CSV cache, and can be safely rerun. On an ordinary laptop, it should finish in minutes rather than tens of minutes; if input sizes exceed the safety limits, it stops with a clear error instead of appearing to hang.


In [ ]:
# =========================================================
# 12. Fast door-to-door network accessibility
# Chunked SciPy sparse Dijkstra — VSCode-safe
# =========================================================

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra as sparse_dijkstra
from time import perf_counter

CHAPTER12_CACHE = DATA_DIR / "chapter12_network_access.csv"
DIJKSTRA_CHUNK_SIZE = 24
MAX_GRAPH_NODES = 150_000
MAX_UNIQUE_ORIGIN_NODES = 10_000

required_origin_fields = {"h3_id", "network_node", "snap_distance_m"}
required_store_fields = {
    "license_number",
    "network_node",
    "snap_distance_m",
    "square_footage",
}

missing_origins = required_origin_fields - set(h3_origin_points.columns)
missing_stores = required_store_fields - set(stores_m.columns)
if missing_origins:
    raise KeyError(f"h3_origin_points is missing: {sorted(missing_origins)}")
if missing_stores:
    raise KeyError(f"stores_m is missing: {sorted(missing_stores)}")

# Keep one record per licensed store so counts are consistent with Chapter 13.
stores_access = (
    stores_m.dropna(subset=["network_node"])
    .sort_values("snap_distance_m")
    .drop_duplicates("license_number")
    .copy()
)
stores_access["square_footage_clean"] = (
    pd.to_numeric(stores_access["square_footage"], errors="coerce")
    .fillna(0)
    .clip(lower=0)
)

origins_access = h3_origin_points.dropna(subset=["network_node"]).copy()

node_list = list(G_walk.nodes())
node_to_position = {node: position for position, node in enumerate(node_list)}

origins_access = origins_access[
    origins_access["network_node"].isin(node_to_position)
].copy()
stores_access = stores_access[
    stores_access["network_node"].isin(node_to_position)
].copy()

unique_origin_nodes = origins_access["network_node"].drop_duplicates().tolist()
unique_store_nodes = stores_access["network_node"].drop_duplicates().tolist()

print(f"Graph nodes: {len(node_list):,}")
print(f"Graph edges: {G_walk.number_of_edges():,}")
print(f"H3 origins: {len(origins_access):,}")
print(f"Unique origin nodes: {len(unique_origin_nodes):,}")
print(f"Licensed stores: {len(stores_access):,}")
print(f"Unique store nodes: {len(unique_store_nodes):,}")

if not origins_access.empty and not stores_access.empty:
    if len(node_list) > MAX_GRAPH_NODES:
        raise RuntimeError(
            f"Graph has {len(node_list):,} nodes, above the safety limit "
            f"of {MAX_GRAPH_NODES:,}. Reduce NETWORK_BUFFER_M or the study area."
        )
    if len(unique_origin_nodes) > MAX_UNIQUE_ORIGIN_NODES:
        raise RuntimeError(
            f"There are {len(unique_origin_nodes):,} unique origins, above the "
            f"safety limit of {MAX_UNIQUE_ORIGIN_NODES:,}. Increase H3 cell size."
        )
else:
    raise ValueError("No valid origins or stores remain after network-node filtering.")

# Build the directed sparse matrix using the minimum length among parallel edges.
# networkx.to_scipy_sparse_array sums parallel edges, which is not appropriate here.
edge_minimum = {}
for u, v, edge_data in G_walk.edges(data=True):
    length = float(edge_data.get("length", np.inf))
    if not np.isfinite(length) or length < 0:
        continue
    key = (node_to_position[u], node_to_position[v])
    previous = edge_minimum.get(key)
    if previous is None or length < previous:
        edge_minimum[key] = length

edge_rows = np.fromiter((key[0] for key in edge_minimum), dtype=np.int64)
edge_cols = np.fromiter((key[1] for key in edge_minimum), dtype=np.int64)
edge_values = np.fromiter(edge_minimum.values(), dtype=float)

graph_matrix = csr_matrix(
    (edge_values, (edge_rows, edge_cols)),
    shape=(len(node_list), len(node_list)),
)

origin_node_positions = np.array(
    [node_to_position[node] for node in unique_origin_nodes], dtype=np.int64
)
store_node_positions_unique = np.array(
    [node_to_position[node] for node in unique_store_nodes], dtype=np.int64
)
store_node_to_unique_position = {
    node: position for position, node in enumerate(unique_store_nodes)
}
store_to_unique_node = np.array(
    [store_node_to_unique_position[node] for node in stores_access["network_node"]],
    dtype=np.int64,
)

store_snap = stores_access["snap_distance_m"].to_numpy(dtype=float)
store_sqft = stores_access["square_footage_clean"].to_numpy(dtype=float)
origin_rows_by_node = origins_access.groupby("network_node").indices

records = []
start_time = perf_counter()
total_batches = math.ceil(len(unique_origin_nodes) / DIJKSTRA_CHUNK_SIZE)

for batch_number, batch_start in enumerate(
    range(0, len(unique_origin_nodes), DIJKSTRA_CHUNK_SIZE), start=1
):
    batch_end = min(batch_start + DIJKSTRA_CHUNK_SIZE, len(unique_origin_nodes))
    batch_nodes = unique_origin_nodes[batch_start:batch_end]
    batch_positions = origin_node_positions[batch_start:batch_end]

    # Returned matrix is batch_size × all graph nodes. The batch limit keeps
    # memory bounded; immediately retain only unique store-node columns.
    graph_distances = sparse_dijkstra(
        graph_matrix,
        directed=True,
        indices=batch_positions,
        limit=float(ACCESS_CUTOFF_M),
        return_predecessors=False,
    )
    if graph_distances.ndim == 1:
        graph_distances = graph_distances[np.newaxis, :]
    distances_to_unique_store_nodes = graph_distances[:, store_node_positions_unique]
    del graph_distances

    for local_position, origin_node in enumerate(batch_nodes):
        graph_to_each_store = distances_to_unique_store_nodes[
            local_position, store_to_unique_node
        ]
        for origin_dataframe_position in origin_rows_by_node[origin_node]:
            origin_row = origins_access.iloc[origin_dataframe_position]
            total_distances = (
                float(origin_row["snap_distance_m"])
                + graph_to_each_store
                + store_snap
            )
            reachable = np.isfinite(total_distances) & (
                total_distances <= ACCESS_CUTOFF_M + 1e-9
            )
            reachable_distances = total_distances[reachable]
            reachable_sqft = store_sqft[reachable]
            decay = np.exp(-reachable_distances / GRAVITY_DECAY_M)

            records.append(
                {
                    "h3_id": origin_row["h3_id"],
                    "network_store_count_10min": int(reachable.sum()),
                    "known_store_count_10min": int((reachable_sqft > 0).sum()),
                    "known_sqft_10min": float(reachable_sqft.sum()),
                    "gravity_store_score": float(decay.sum()),
                    "gravity_sqft_score": float(
                        ((reachable_sqft / 1000) * decay).sum()
                    ),
                    "mean_reachable_store_distance_m": (
                        float(reachable_distances.mean())
                        if reachable_distances.size
                        else np.nan
                    ),
                }
            )

    elapsed = perf_counter() - start_time
    print(
        f"Batch {batch_number}/{total_batches} complete — "
        f"{batch_end:,}/{len(unique_origin_nodes):,} unique origins — "
        f"{elapsed:.1f} seconds elapsed"
    )

node_access = pd.DataFrame.from_records(records)
if node_access.empty:
    raise RuntimeError("Chapter 12 produced no accessibility records.")
if node_access["h3_id"].duplicated().any():
    raise RuntimeError("Chapter 12 produced duplicate h3_id records.")

node_access.to_csv(CHAPTER12_CACHE, index=False)

chapter12_fields = [
    "network_store_count_10min",
    "known_store_count_10min",
    "known_sqft_10min",
    "gravity_store_score",
    "gravity_sqft_score",
    "mean_reachable_store_distance_m",
]
access = access.drop(
    columns=[field for field in chapter12_fields if field in access.columns],
    errors="ignore",
)
access = access.merge(node_access, on="h3_id", how="left", validate="one_to_one")

elapsed = perf_counter() - start_time
print(f"Chapter 12 completed in {elapsed:.1f} seconds.")
print(f"Cached results: {CHAPTER12_CACHE}")
display(node_access.head())


## 13. Compare circular buffers with the network

A circular 800-meter buffer is equivalent to asking which stores fall within 800 meters of each origin in projected coordinate space. This implementation uses two `cKDTree` spatial indexes and a sparse origin–store pair matrix. It performs no polygon buffering, no spatial join, and no per-origin Pandas DataFrame construction. Store records are deduplicated once by license number before the spatial search.


In [ ]:
# =========================================================
# 13. Compare circular buffers with the network
# Vectorized sparse-pair implementation — VSCode-safe
# =========================================================

from time import perf_counter
from scipy.spatial import cKDTree

chapter13_start = perf_counter()

# ---------------------------------------------------------
# 1. Validate required inputs
# ---------------------------------------------------------
required_origin_columns = {"h3_id", "geometry"}
required_store_columns = {"license_number", "square_footage", "geometry"}
required_access_columns = {"h3_id", "network_store_count_10min"}

missing_origin = required_origin_columns - set(h3_origin_points.columns)
missing_store = required_store_columns - set(stores_m.columns)
missing_access = required_access_columns - set(access.columns)

if missing_origin:
    raise KeyError(f"h3_origin_points is missing: {sorted(missing_origin)}")
if missing_store:
    raise KeyError(f"stores_m is missing: {sorted(missing_store)}")
if missing_access:
    raise KeyError(
        f"access is missing: {sorted(missing_access)}. Run Chapters 9–12 first."
    )
if h3_origin_points.crs is None or stores_m.crs is None:
    raise ValueError("Origins and stores must both have a CRS.")
if h3_origin_points.crs != stores_m.crs:
    raise ValueError(
        f"CRS mismatch: origins={h3_origin_points.crs}, stores={stores_m.crs}"
    )
if h3_origin_points.crs.is_geographic:
    raise ValueError(
        "Chapter 13 requires a projected CRS measured in meters, such as EPSG:32618."
    )

# ---------------------------------------------------------
# 2. Clean and deduplicate inputs once
# ---------------------------------------------------------
origins_for_buffer = h3_origin_points[["h3_id", "geometry"]].copy()
origins_for_buffer = origins_for_buffer[
    origins_for_buffer["h3_id"].notna()
    & origins_for_buffer.geometry.notna()
    & ~origins_for_buffer.geometry.is_empty
].drop_duplicates("h3_id").reset_index(drop=True)

stores_for_buffer = stores_m[
    ["license_number", "square_footage", "geometry"]
].copy()
stores_for_buffer = stores_for_buffer[
    stores_for_buffer["license_number"].notna()
    & stores_for_buffer.geometry.notna()
    & ~stores_for_buffer.geometry.is_empty
].copy()
stores_for_buffer["license_number"] = (
    stores_for_buffer["license_number"].astype(str).str.strip()
)
stores_for_buffer["square_footage_clean"] = (
    pd.to_numeric(stores_for_buffer["square_footage"], errors="coerce")
    .fillna(0.0)
    .clip(lower=0.0)
)

# A license is counted once. Keep one geometry and the largest known area per license.
stores_for_buffer = (
    stores_for_buffer.sort_values("square_footage_clean", ascending=False)
    .drop_duplicates("license_number", keep="first")
    .reset_index(drop=True)
)

n_origins = len(origins_for_buffer)
n_stores = len(stores_for_buffer)

print(f"Chapter 13 input — origins: {n_origins:,}; unique stores: {n_stores:,}")

if n_origins == 0:
    raise ValueError("No valid H3 origin points are available.")
if n_stores == 0:
    raise ValueError("No valid stores are available.")

# Fail fast if upstream data unexpectedly exploded instead of appearing to freeze.
if n_origins > 100_000:
    raise RuntimeError(
        f"Unexpectedly large origin table ({n_origins:,} rows). "
        "Check earlier joins for duplicated h3_id values."
    )
if n_stores > 100_000:
    raise RuntimeError(
        f"Unexpectedly large store table ({n_stores:,} rows). "
        "Check the study-area filter before Chapter 13."
    )

# ---------------------------------------------------------
# 3. Build coordinate arrays and spatial indexes
# ---------------------------------------------------------
origin_coordinates = np.column_stack(
    (
        origins_for_buffer.geometry.x.to_numpy(dtype=np.float64),
        origins_for_buffer.geometry.y.to_numpy(dtype=np.float64),
    )
)
store_coordinates = np.column_stack(
    (
        stores_for_buffer.geometry.x.to_numpy(dtype=np.float64),
        stores_for_buffer.geometry.y.to_numpy(dtype=np.float64),
    )
)

origin_tree = cKDTree(origin_coordinates)
store_tree = cKDTree(store_coordinates)

print("Building sparse origin–store pairs within the Euclidean threshold...")
search_start = perf_counter()

# The COO matrix stores only pairs whose distance is <= ACCESS_CUTOFF_M.
# No full origin × store distance matrix is created.
pairs = origin_tree.sparse_distance_matrix(
    store_tree,
    max_distance=float(ACCESS_CUTOFF_M),
    output_type="coo_matrix",
)

print(
    f"Spatial search complete: {pairs.nnz:,} qualifying pairs "
    f"in {perf_counter() - search_start:.2f} seconds."
)

# ---------------------------------------------------------
# 4. Vectorized aggregation — no per-origin DataFrames
# ---------------------------------------------------------
# Because stores were deduplicated by license before the search, each COO pair
# represents one unique licensed store available to one origin.
euclidean_counts = np.bincount(
    pairs.row,
    minlength=n_origins,
).astype(np.int64)

store_sqft = stores_for_buffer["square_footage_clean"].to_numpy(dtype=np.float64)
euclidean_sqft = np.bincount(
    pairs.row,
    weights=store_sqft[pairs.col],
    minlength=n_origins,
).astype(np.float64)

euclidean_access = pd.DataFrame(
    {
        "h3_id": origins_for_buffer["h3_id"].to_numpy(),
        "euclidean_store_count_10min": euclidean_counts,
        "euclidean_known_sqft_10min": euclidean_sqft,
    }
)

# ---------------------------------------------------------
# 5. Merge safely so the cell can be rerun
# ---------------------------------------------------------
replace_fields = [
    "euclidean_store_count_10min",
    "euclidean_known_sqft_10min",
    "buffer_overcount",
    "buffer_overcount_pct",
]
access = access.drop(
    columns=[field for field in replace_fields if field in access.columns],
    errors="ignore",
)

# Guarantee one row per h3_id before a one-to-one merge.
if access["h3_id"].duplicated().any():
    duplicate_count = int(access["h3_id"].duplicated(keep=False).sum())
    raise RuntimeError(
        f"access contains {duplicate_count:,} rows with duplicated h3_id values. "
        "Fix the earlier merge before running Chapter 13."
    )

access = access.merge(
    euclidean_access,
    on="h3_id",
    how="left",
    validate="one_to_one",
)

access["euclidean_store_count_10min"] = (
    access["euclidean_store_count_10min"].fillna(0).astype(np.int64)
)
access["euclidean_known_sqft_10min"] = (
    access["euclidean_known_sqft_10min"].fillna(0.0).astype(float)
)
access["network_store_count_10min"] = (
    pd.to_numeric(access["network_store_count_10min"], errors="coerce")
    .fillna(0)
    .astype(np.int64)
)

access["buffer_overcount"] = (
    access["euclidean_store_count_10min"]
    - access["network_store_count_10min"]
)
access["buffer_overcount_pct"] = np.where(
    access["euclidean_store_count_10min"] > 0,
    access["buffer_overcount"]
    / access["euclidean_store_count_10min"]
    * 100.0,
    np.nan,
)

# Save a lightweight cache so later chapters can resume without recomputing.
chapter13_cache = DATA_DIR / "chapter13_euclidean_access.csv"
euclidean_access.to_csv(chapter13_cache, index=False)

print(f"Chapter 13 completed in {perf_counter() - chapter13_start:.2f} seconds.")
print(f"Saved cache: {chapter13_cache}")
print(
    "Cells where Euclidean buffers overcount:",
    f"{int((access['buffer_overcount'] > 0).sum()):,}",
)
print(
    "Cells where both methods agree:",
    f"{int((access['buffer_overcount'] == 0).sum()):,}",
)
print(
    "Cells where network count is larger:",
    f"{int((access['buffer_overcount'] < 0).sum()):,}",
)

display(
    access[
        [
            "h3_id",
            "euclidean_store_count_10min",
            "network_store_count_10min",
            "buffer_overcount",
            "buffer_overcount_pct",
        ]
    ]
    .sort_values("buffer_overcount", ascending=False)
    .head(10)
)


In [ ]:
from IPython.display import Image
plot_data = access[
    ["euclidean_store_count_10min", "network_store_count_10min"]
].copy()
plot_data = plot_data.apply(pd.to_numeric, errors="coerce")
plot_data = plot_data.replace([np.inf, -np.inf], np.nan).dropna()

if plot_data.empty:
    print("No valid rows are available for the comparison chart.")
else:
    maximum = float(max(plot_data.max().max(), 1))
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(
        plot_data["euclidean_store_count_10min"],
        plot_data["network_store_count_10min"],
        s=18,
        alpha=0.5,
        edgecolors="none",
        rasterized=True,
    )
    ax.plot(
        [0, maximum],
        [0, maximum],
        linestyle="--",
        color="black",
        linewidth=1,
    )
    ax.set_xlim(0, maximum * 1.03)
    ax.set_ylim(0, maximum * 1.03)
    ax.set_xlabel("Stores inside 800 m Euclidean buffer")
    ax.set_ylabel("Stores reachable within 800 m door-to-door distance")
    ax.set_title("Circular-buffer access versus corrected network access")
    fig.tight_layout()
    output_path = FIGURE_DIR / "09_buffer_vs_network_access.png"
    fig.savefig(output_path, dpi=150, facecolor="white")
    print(f"Saved figure to: {output_path}")
    plt.close(fig)
    display(Image(filename=str(output_path), width=700))


## 14. Create transparent pressure indicators and test priority sensitivity

Store pressure and known-capacity pressure remain separate because store counts and recorded floor area answer different questions. Four normalized components are created:

- residential demand — higher units receive a higher score;
- nearest network distance — longer distance receives a higher score;
- store scarcity — fewer reachable stores receive a higher score; and
- buffer overcount — larger Euclidean overstatement receives a higher score.

The notebook computes balanced, demand-focused, and access-focused priority indices. It then reports Spearman rank correlation, top-15 overlap, and each cell's rank spread. The final `priority_index` is the mean of the three scenarios rather than one arbitrary formula.

In [ ]:
access["store_pressure"] = (
    access["residential_units"] / (access["network_store_count_10min"] + 1)
)
access["known_capacity_pressure"] = (
    access["residential_units"] / (access["known_sqft_10min"] / 1000 + 1)
)

access["demand_percentile"] = percentile_high(access["residential_units"])
access["distance_percentile"] = percentile_high(access["network_m"])
access["scarcity_percentile"] = percentile_low(access["network_store_count_10min"])
access["overcount_percentile"] = percentile_high(access["buffer_overcount"])

priority_columns = []
for scenario_name, weights in PRIORITY_WEIGHT_SCENARIOS.items():
    column = f"priority_{scenario_name}"
    access[column] = 100 * (
        weights["demand"] * access["demand_percentile"]
        + weights["distance"] * access["distance_percentile"]
        + weights["scarcity"] * access["scarcity_percentile"]
        + weights["overcount"] * access["overcount_percentile"]
    )
    priority_columns.append(column)
    access[f"rank_{scenario_name}"] = access[column].rank(
        method="min", ascending=False
    )

access["priority_index"] = access[priority_columns].mean(axis=1)
rank_columns = [f"rank_{name}" for name in PRIORITY_WEIGHT_SCENARIOS]
access["priority_rank_min"] = access[rank_columns].min(axis=1)
access["priority_rank_max"] = access[rank_columns].max(axis=1)
access["priority_rank_spread"] = (
    access["priority_rank_max"] - access["priority_rank_min"]
)

priority_rank_correlation = access[priority_columns].corr(method="spearman")

scenario_top_ids = {
    name: set(
        access.nlargest(TOP_PRIORITY_COUNT, f"priority_{name}")["h3_id"]
    )
    for name in PRIORITY_WEIGHT_SCENARIOS
}
scenario_names = list(PRIORITY_WEIGHT_SCENARIOS)
top_overlap = pd.DataFrame(index=scenario_names, columns=scenario_names, dtype=float)
for left in scenario_names:
    for right in scenario_names:
        union = scenario_top_ids[left] | scenario_top_ids[right]
        top_overlap.loc[left, right] = (
            len(scenario_top_ids[left] & scenario_top_ids[right]) / len(union)
            if union
            else np.nan
        )

print("Spearman correlation among priority scenarios")
display(priority_rank_correlation.round(3))
print("Jaccard overlap among top-priority cell sets")
display(top_overlap.round(3))

priority_rank_correlation.to_csv(TABLE_DIR / "priority_rank_correlation.csv")
top_overlap.to_csv(TABLE_DIR / "priority_top15_overlap.csv")

## 15. Stress-test multiple large-store closure scenarios

A single “largest store” scenario can overstate the importance of one incomplete attribute. This version selects the three largest records with known positive floor area and runs a separate corrected door-to-door nearest-store calculation after removing each store. The results are not predictions of closure; they are comparable stress tests that reveal dependence within the model.

In [ ]:
known_size_stores = stores_m[stores_m["square_footage"].fillna(0).gt(0)].copy()
if known_size_stores.empty:
    print(
        "No usable square-footage values were returned. "
        "Closure scenarios will use the three stores with the largest ten-minute catchments."
    )
    catchment_counts = []
    for store in stores_m.itertuples():
        catchment_counts.append((store.license_number, 0))
    closure_candidates = stores_m.head(min(CLOSURE_SCENARIO_COUNT, len(stores_m))).copy()
else:
    closure_candidates = (
        known_size_stores.sort_values("square_footage", ascending=False)
        .drop_duplicates("network_node")
        .head(min(CLOSURE_SCENARIO_COUNT, len(known_size_stores)))
        .copy()
    )

if closure_candidates.empty:
    raise ValueError("No stores are available for closure stress tests.")

closure_summary_rows = []
closure_scenario_fields = []
for scenario_number, closed_store in enumerate(
    closure_candidates.itertuples(), start=1
):
    remaining_stores = stores_m[
        stores_m["license_number"] != closed_store.license_number
    ].copy()
    if remaining_stores.empty:
        continue
    scenario_cost, scenario_nearest, _ = multi_source_store_dijkstra(
        G_walk,
        remaining_stores,
    )

    scenario_distance_field = f"closure_{scenario_number}_network_m"
    scenario_increase_field = f"closure_{scenario_number}_increase_min"
    scenario_affected_field = f"closure_{scenario_number}_affected"

    scenario_distance_by_h3 = {
        row.h3_id: row.snap_distance_m + scenario_cost.get(row.network_node, np.nan)
        for row in h3_origin_points.itertuples()
    }
    access[scenario_distance_field] = access["h3_id"].map(scenario_distance_by_h3)
    access[scenario_increase_field] = (
        access[scenario_distance_field] - access["network_m"]
    ) / WALKING_SPEED_M_PER_MIN
    access[scenario_affected_field] = access[scenario_increase_field].fillna(0).gt(1e-6)

    affected_units = access.loc[
        access[scenario_affected_field], "residential_units"
    ].sum()
    unit_weighted_increase = weighted_average(
        access[scenario_increase_field].clip(lower=0),
        access["residential_units"],
    )
    closure_summary_rows.append(
        {
            "scenario": scenario_number,
            "license_number": closed_store.license_number,
            "store_name": closed_store.store_name,
            "address": closed_store.address,
            "recorded_square_footage": closed_store.square_footage,
            "affected_residential_units": affected_units,
            "unit_weighted_increase_min": unit_weighted_increase,
            "maximum_increase_min": access[scenario_increase_field].max(),
        }
    )
    closure_scenario_fields.append(scenario_increase_field)

closure_summary = pd.DataFrame(closure_summary_rows)
if not closure_scenario_fields:
    raise ValueError("Closure scenarios could not be calculated.")

access["closure_increase_min"] = access[closure_scenario_fields].max(axis=1)
access["closure_increase_m"] = access["closure_increase_min"] * WALKING_SPEED_M_PER_MIN
access["closure_affected"] = access["closure_increase_min"].fillna(0).gt(1e-6)
affected_columns = [
    column for column in access.columns if re.fullmatch(r"closure_\d+_affected", column)
]
access["closure_scenario_count_affected"] = access[affected_columns].sum(axis=1)

print("Closure stress-test summary")
display(closure_summary.round(2))
closure_summary.to_csv(TABLE_DIR / "closure_scenarios.csv", index=False)


## 16. Join metrics back to H3 geometry

In [ ]:
access_cells = h3_demand.merge(access, on="h3_id", how="inner", suffixes=("", "_access"))
access_cells = gpd.GeoDataFrame(access_cells, geometry="geometry", crs=h3_demand.crs)

print(f"Final analytical cells: {len(access_cells):,}")
print(f"Residential units in final analytical cells: {access_cells['residential_units'].sum():,.2f}")
access_cells.head()

## 17. Map the principal findings

These maps separate distinct questions rather than combining every variable into one composite graphic.

In [ ]:
plot_metric(
    access_cells,
    "residential_units",
    "Residential demand allocated to H3 cells",
    "Allocated residential units",
    cmap="viridis",
    filename="10_residential_units.png",
)

In [ ]:
plot_metric(
    access_cells,
    "network_m",
    "Corrected door-to-door distance to the nearest store",
    "Distance (m)",
    cmap="magma",
    filename="11_network_distance.png",
)

In [ ]:
plot_metric(
    access_cells,
    "detour_index",
    "Where the walking network creates detours",
    "Door-to-door / Euclidean distance",
    cmap="magma",
    filename="12_detour_index.png",
)

In [ ]:
plot_metric(
    access_cells,
    "network_store_count_10min",
    "Stores reachable within ten door-to-door minutes",
    "Store count",
    cmap="viridis_r",
    filename="13_store_count_10min.png",
)

In [ ]:
plot_metric(
    access_cells,
    "buffer_overcount",
    "Where circular buffers overstate store access",
    "Euclidean count minus network count",
    cmap="Reds",
    filename="14_buffer_overcount.png",
)

In [ ]:
plot_metric(
    access_cells,
    "priority_index",
    "Robust priority index averaged across three scenarios",
    "Priority index",
    cmap="YlOrRd",
    filename="15_robust_priority_index.png",
)

In [ ]:
plot_metric(
    access_cells,
    "closure_increase_min",
    "Maximum impact across three closure stress tests",
    "Additional walking minutes",
    cmap="Reds",
    filename="16_closure_impact.png",
)

In [ ]:
plot_metric(
    access_cells,
    "priority_rank_spread",
    "Priority-ranking sensitivity across three weighting scenarios",
    "Rank spread (cells)",
    cmap="Purples",
    filename="17_priority_rank_sensitivity.png",
)

## 18. Summarize results with residential-unit weights

Cell-level medians describe the spatial surface, while unit-weighted statistics give greater influence to cells containing more residential demand. Both are reported so low-demand and high-demand cells are not silently conflated.

In [ ]:
summary = pd.Series(
    {
        "H3 cells": len(access_cells),
        "Residential units represented": access_cells["residential_units"].sum(),
        "Median nearest network distance (m)": access_cells["network_m"].median(),
        "Unit-weighted nearest network distance (m)": weighted_average(
            access_cells["network_m"], access_cells["residential_units"]
        ),
        "Median detour index": access_cells["detour_index"].median(),
        "Unit-weighted detour index": weighted_average(
            access_cells["detour_index"], access_cells["residential_units"]
        ),
        "Share of cells with a different nearest store by network": (
            access_cells["nearest_store_changed"].mean()
        ),
        "Share of residential units with a different nearest store by network": weighted_share(
            access_cells["nearest_store_changed"], access_cells["residential_units"]
        ),
        "Median network-reachable stores in ten minutes": (
            access_cells["network_store_count_10min"].median()
        ),
        "Unit-weighted network-reachable stores in ten minutes": weighted_average(
            access_cells["network_store_count_10min"], access_cells["residential_units"]
        ),
        "Median circular-buffer overcount": access_cells["buffer_overcount"].median(),
        "Unit-weighted circular-buffer overcount": weighted_average(
            access_cells["buffer_overcount"], access_cells["residential_units"]
        ),
        "Residential units affected in at least one closure scenario": access_cells.loc[
            access_cells["closure_affected"], "residential_units"
        ].sum(),
        "Median priority rank spread": access_cells["priority_rank_spread"].median(),
    }
)
summary.to_frame("value")

### Results by community district

The dominant community district for each H3 cell is determined by the largest share of proportionally allocated residential units. This table supports interpretation at a recognizable administrative scale without replacing the finer-grained H3 analysis.

In [ ]:
district_rows = []
for district, group in access_cells.groupby("dominant_cd", dropna=False):
    district_rows.append(
        {
            "community_district": int(district) if pd.notna(district) else pd.NA,
            "district_name": (
                COMMUNITY_DISTRICT_NAMES.get(int(district), "Unknown")
                if pd.notna(district)
                else "Unknown"
            ),
            "h3_cells": len(group),
            "residential_units": group["residential_units"].sum(),
            "weighted_network_m": weighted_average(
                group["network_m"], group["residential_units"]
            ),
            "weighted_store_count_10min": weighted_average(
                group["network_store_count_10min"], group["residential_units"]
            ),
            "weighted_buffer_overcount": weighted_average(
                group["buffer_overcount"], group["residential_units"]
            ),
            "weighted_priority_index": weighted_average(
                group["priority_index"], group["residential_units"]
            ),
            "closure_affected_units": group.loc[
                group["closure_affected"], "residential_units"
            ].sum(),
        }
    )

district_summary = pd.DataFrame(district_rows).sort_values("community_district")
district_summary.to_csv(TABLE_DIR / "district_summary.csv", index=False)
district_summary.round(2)

### Automatically generated findings

This cell converts the actual run outputs into a concise draft. It should be edited for final presentation after checking the mapped extreme cases and field context.

In [ ]:
worst_distance = district_summary.nlargest(1, "weighted_network_m").iloc[0]
lowest_access = district_summary.nsmallest(1, "weighted_store_count_10min").iloc[0]
highest_overcount = district_summary.nlargest(1, "weighted_buffer_overcount").iloc[0]
most_affected_closure = closure_summary.nlargest(1, "affected_residential_units").iloc[0]

findings_markdown = f"""
## Key findings from the completed run

1. The unit-weighted door-to-door distance to the nearest licensed retail food store is **{summary['Unit-weighted nearest network distance (m)']:.0f} meters**, while the median detour index is **{summary['Median detour index']:.2f}**.
2. The nearest store changes when network structure is considered for **{summary['Share of residential units with a different nearest store by network']:.1%} of represented residential units**.
3. Circular 800-meter buffers overcount access by a unit-weighted average of **{summary['Unit-weighted circular-buffer overcount']:.1f} stores per demand cell**.
4. **{worst_distance['district_name']}** has the largest unit-weighted nearest-store distance among the four study districts, while **{lowest_access['district_name']}** has the lowest weighted ten-minute store count.
5. The greatest district-level Euclidean overstatement occurs in **{highest_overcount['district_name']}**.
6. In the strongest modeled closure stress test, removing **{most_affected_closure['store_name']}** affects cells representing approximately **{most_affected_closure['affected_residential_units']:,.0f} residential units**.
7. The median rank spread across the three priority-weight scenarios is **{summary['Median priority rank spread']:.0f} places**, showing how strongly the priority map depends on explicit value choices.

These findings describe modeled accessibility, not food security. Price, quality, opening hours, entrances, slope, safety, disability, and lived experience require field validation.
"""
display(Markdown(findings_markdown))
(DATA_DIR / "generated_findings.md").write_text(findings_markdown, encoding="utf-8")

## 19. Identify robust priority cells for further investigation

The table is not a final list of “food deserts.” It identifies cells where demand, network distance, store scarcity, and buffer overcount overlap across multiple weighting scenarios. Rank spread helps distinguish robust priorities from cells that rise or fall sharply when the weights change.

In [ ]:
priority_cells = access_cells.nlargest(TOP_PRIORITY_COUNT, "priority_index")[
    [
        "h3_id",
        "dominant_cd",
        "district_name",
        "residential_units",
        "network_m",
        "walk_min_nearest",
        "network_store_count_10min",
        "buffer_overcount",
        "known_capacity_pressure",
        "priority_balanced",
        "priority_demand_focused",
        "priority_access_focused",
        "priority_index",
        "priority_rank_spread",
        "closure_increase_min",
    ]
].copy()
priority_cells.to_csv(TABLE_DIR / "priority_cells.csv", index=False)
priority_cells.round(2)

## 20. Inspect disagreement, extreme cases, and representative routes

Large detour ratios and buffer overcounts may reveal meaningful barriers, but they may also reveal geocoding errors, disconnected graph components, or inappropriate snapping. Extreme values are mapped, and three representative routes are reconstructed with the origin and destination connector segments visible.

In [ ]:
extreme_cases = access_cells.nlargest(20, "detour_index")[
    [
        "h3_id",
        "dominant_cd",
        "district_name",
        "residential_units",
        "euclidean_m",
        "network_m",
        "network_minus_euclidean_m",
        "detour_index",
        "snap_distance_m",
        "network_store_snap_m",
        "euclidean_store_name",
        "network_store_name",
        "buffer_overcount",
    ]
]
extreme_cases.to_csv(TABLE_DIR / "extreme_detour_cases.csv", index=False)
extreme_cases.round(2)

In [ ]:
extreme_ids = extreme_cases["h3_id"].tolist()
ax = network_edges.plot(figsize=(10, 13), color="#bdbdbd", linewidth=0.15)
access_cells.plot(ax=ax, color="#efefef", edgecolor="white", linewidth=0.1)
access_cells[access_cells["h3_id"].isin(extreme_ids)].plot(
    ax=ax,
    column="detour_index",
    cmap="magma",
    legend=True,
    edgecolor="black",
    linewidth=0.4,
)
stores_m.plot(ax=ax, color="blue", markersize=7, alpha=0.5)
ax.set_title("Twenty largest corrected detour-index cells")
ax.set_axis_off()
save_current_figure("18_extreme_detour_cells.png")
plt.show()

route_case_ids = {
    "Maximum detour": access_cells.nlargest(1, "detour_index")["h3_id"].iloc[0],
    "Maximum robust priority": access_cells.nlargest(1, "priority_index")["h3_id"].iloc[0],
    "Median network distance": access_cells.iloc[
        (access_cells["network_m"] - access_cells["network_m"].median()).abs().argsort()[:1]
    ]["h3_id"].iloc[0],
}

origin_lookup = h3_origin_points.set_index("h3_id")
route_rows = []
for route_label, h3_id in route_case_ids.items():
    origin = origin_lookup.loc[h3_id]
    store_license = nearest_store_license.get(origin["network_node"])
    store = store_lookup.loc[store_license]
    node_path = reconstruct_node_path(origin["network_node"], toward_store)
    network_coordinates = [network_nodes.loc[node].geometry.coords[0] for node in node_path]
    coordinates = [origin.geometry.coords[0], *network_coordinates, store.geometry.coords[0]]
    cleaned_coordinates = [coordinates[0]]
    for coordinate in coordinates[1:]:
        if coordinate != cleaned_coordinates[-1]:
            cleaned_coordinates.append(coordinate)
    route_rows.append(
        {
            "route_label": route_label,
            "h3_id": h3_id,
            "store_name": store["store_name"],
            "network_m": access_cells.set_index("h3_id").loc[h3_id, "network_m"],
            "geometry": LineString(cleaned_coordinates),
        }
    )

representative_routes = gpd.GeoDataFrame(route_rows, geometry="geometry", crs=ANALYSIS_CRS)
ax = network_edges.plot(figsize=(10, 13), color="#d0d0d0", linewidth=0.15)
study_core_m.boundary.plot(ax=ax, color="black", linewidth=0.8)
representative_routes.plot(ax=ax, column="route_label", linewidth=3, legend=True)
stores_m.plot(ax=ax, color="#17365d", markersize=10)
ax.set_title("Representative corrected door-to-door routes")
ax.set_axis_off()
save_current_figure("19_representative_routes.png")
plt.show()

## 21. Optional interactive Lonboard view

This cell follows the tutorials' use of Lonboard for interactive spatial inspection. It is optional because the final project also exports a standalone MapLibre interface.

In [ ]:
try:
    from lonboard import Map, PolygonLayer, ScatterplotLayer
    from lonboard.colormap import apply_continuous_cmap
    from matplotlib.colors import Normalize
    from palettable.colorbrewer.sequential import YlOrRd_9

    interactive_cells = access_cells.to_crs("EPSG:4326").copy()
    values = interactive_cells["priority_index"].fillna(0).to_numpy()
    colors = apply_continuous_cmap(
        values,
        YlOrRd_9,
        normalizer=Normalize(values.min(), values.max()),
    )

    cell_layer = PolygonLayer.from_geopandas(
        interactive_cells[
            [
                "h3_id", "district_name", "residential_units", "network_m",
                "network_store_count_10min", "buffer_overcount", "priority_index",
                "priority_rank_spread", "closure_increase_min", "geometry",
            ]
        ],
        get_fill_color=colors,
        get_line_color=[255, 255, 255, 180],
        line_width_min_pixels=0.5,
        pickable=True,
        opacity=0.72,
    )

    store_layer = ScatterplotLayer.from_geopandas(
        stores_m.to_crs("EPSG:4326")[["store_name", "store_size", "geometry"]],
        get_radius=20,
        get_fill_color=[20, 40, 90, 190],
        pickable=True,
    )

    m = Map([cell_layer, store_layer])
    m
except Exception as exc:
    print(f"Optional Lonboard preview skipped: {exc}")
    print("The analytical workflow and MapLibre export can continue normally.")

## 22. Export analytical data for MapLibre and submission review

The web map uses WGS84 GeoJSON. Interpretable fields from the corrected calculations, priority scenarios, allocation comparison, routes, and closure stress tests are retained. Summary JSON and CSV tables allow the web interface and reviewer to inspect the same results shown in the notebook.

In [ ]:
web_cells = access_cells.to_crs("EPSG:4326").copy()
cell_fields = [
    "h3_id", "dominant_cd", "district_name", "residential_units",
    "centroid_residential_units", "allocation_difference_units",
    "allocation_difference_pct", "euclidean_m", "network_m",
    "network_minus_euclidean_m", "walk_min_nearest", "detour_index",
    "nearest_store_changed", "network_store_count_10min", "known_sqft_10min",
    "euclidean_store_count_10min", "buffer_overcount", "buffer_overcount_pct",
    "store_pressure", "known_capacity_pressure", "priority_balanced",
    "priority_demand_focused", "priority_access_focused", "priority_index",
    "priority_rank_spread", "closure_increase_m", "closure_increase_min",
    "closure_scenario_count_affected", "geometry",
]
web_cells = web_cells[cell_fields]

numeric_fields = web_cells.select_dtypes(include="number").columns
web_cells[numeric_fields] = web_cells[numeric_fields].round(2)

web_stores = stores_m.to_crs("EPSG:4326")[[
    "license_number", "store_name", "address", "store_size", "square_footage",
    "estab_type", "operation_type", "snap_distance_m", "geometry",
]].copy()

closure_points = closure_candidates.to_crs("EPSG:4326").copy()
closure_points["scenario"] = range(1, len(closure_points) + 1)
closure_points = closure_points[[
    "scenario", "license_number", "store_name", "address", "square_footage", "geometry"
]]
web_routes = representative_routes.to_crs("EPSG:4326")

web_cells.to_file(WEB_DATA_DIR / "access_cells.geojson", driver="GeoJSON")
web_stores.to_file(WEB_DATA_DIR / "stores.geojson", driver="GeoJSON")
study_core.to_file(WEB_DATA_DIR / "study_core.geojson", driver="GeoJSON")
study_boundary.to_file(WEB_DATA_DIR / "study_boundary.geojson", driver="GeoJSON")
closure_points.to_file(WEB_DATA_DIR / "closure_stores.geojson", driver="GeoJSON")
web_routes.to_file(WEB_DATA_DIR / "representative_routes.geojson", driver="GeoJSON")
priority_cells.to_csv(TABLE_DIR / "priority_cells.csv", index=False)

RUN_METADATA["run_completed_utc"] = datetime.now(timezone.utc).isoformat()
RUN_METADATA["final_analytical_cells"] = int(len(access_cells))
RUN_METADATA["final_residential_units"] = float(access_cells["residential_units"].sum())
RUN_METADATA["figures_generated"] = sorted(path.name for path in FIGURE_DIR.glob("*.png"))
RUN_METADATA["tables_generated"] = sorted(path.name for path in TABLE_DIR.glob("*.csv"))
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2), encoding="utf-8")

summary_payload = {
    "summary": {key: (None if pd.isna(value) else float(value)) for key, value in summary.items()},
    "closure_scenarios": closure_summary.replace({np.nan: None}).to_dict(orient="records"),
    "priority_weights": PRIORITY_WEIGHT_SCENARIOS,
    "run_metadata": RUN_METADATA,
}
(WEB_DATA_DIR / "summary.json").write_text(
    json.dumps(summary_payload, indent=2, default=str),
    encoding="utf-8",
)

print("Exported corrected MapLibre data, summary JSON, figures, and review tables.")

## 23. Bias evaluation and limitations

### Data coverage
- The retail dataset is a licensing snapshot, not a live inventory of open stores.
- Informal vendors, institutional dining, delivery-only businesses, temporary markets, and unlicensed food sources may be absent.
- A store point does not describe affordability, product quality, cultural relevance, opening hours, inventory, or benefit acceptance.
- Store floor area is missing for some records. Counts and known floor area are therefore kept separate.
- The operation-type filter is printed and saved in metadata because classification codes require interpretation.

### Demand representation
- `UnitsRes` counts housing units rather than residents and does not include household size, income, age, disability, or food insecurity.
- Proportional area allocation assumes units are evenly distributed across each tax lot; vertical building organization is not represented.
- H3 resolution creates a modifiable areal unit problem. The centroid comparison reveals one form of allocation sensitivity but does not eliminate it.
- Each cell still uses one representative origin point for network calculations.

### Network representation
- OpenStreetMap coverage and tags may omit pedestrian entrances, stairs, gates, passages, construction, or temporary closures.
- A constant walking speed excludes slope, weather, crossing delay, mobility differences, safety, and the burden of carrying groceries.
- Origin and store snap distances are now included, but snapping remains a modeled approximation.
- The graph retains one connected component, which can exclude disconnected but real pedestrian spaces.

### Analytical choices
- The 800-meter threshold is a modeling assumption rather than a universal definition of access.
- Store count treats stores as comparable; known floor area is incomplete and does not directly measure useful food capacity.
- Priority weights encode values. Three scenarios and rank spread make that sensitivity visible but do not resolve the normative choice.
- Closure scenarios are stress tests, not forecasts of customer behavior, substitution, pricing, or supply-chain response.

### Interpretation protocol
- Extreme detours, large buffer overcounts, unstable rankings, and closure-sensitive cells should be treated as prompts for fieldwork.
- No cell is labeled a “food desert” from these metrics alone.

## 24. Reflection on the method

The most important development from the original network exercise is the shift from finding one route to constructing and questioning an accessibility surface. A shortest-path map can answer where one person should walk, but it does not by itself show how network structure distributes opportunity across a neighborhood. Proportional allocation connects residential tax lots to H3 demand cells while making the aggregation method visible and testable.

The corrected distance calculation also changes the analytical claim. Node-to-node distance can undercount the trip when origins and stores are not located directly on graph nodes. Including both connector distances creates a consistent door-to-door comparison with Euclidean distance and provides a diagnostic when network results appear physically impossible.

The comparison between circular buffers and network access is methodological rather than merely visual. A circle treats distance as continuous and directionless; a pedestrian graph treats access as a sequence of permitted connections. Their disagreement identifies places where conventional proximity graphics need further inspection. The network result is not automatically “reality,” however. Its edges, weights, representative origins, and destinations still encode assumptions.

The sensitivity analysis makes another hidden choice explicit. A priority index changes when residential demand, distance, scarcity, or mapping error receives more weight. Averaging three scenarios produces a robust exploratory surface, while rank spread highlights locations whose apparent importance depends heavily on the researcher's values. That instability is itself a finding.

Finally, the closure analysis reveals the difference between proximity and resilience. A place may have a nearby store under normal conditions yet depend strongly on one destination. Because store floor area is incomplete and does not equal food quality or affordability, the stress tests should guide field investigation rather than claim to predict hardship.

## 25. Further exploration through the following semesters

### Fall semester — situated validation
I will select several robust high-priority cells, unstable high-priority cells, and low-priority comparison cells for field observation. I will record actual entrances, stairs, slopes, crossing delays, store hours, price samples, food categories, and carrying conditions. Observed walking routes will be compared with modeled routes. Particular attention will be paid to Columbia's campus edges, Morningside Park, superblocks, and elevation changes because these conditions may produce differences invisible in the present graph.

### Spring semester — food infrastructure and delivery systems
The project will expand from consumer walking access to the broader food chain. Retail stores will become one node type within a network that includes Hunts Point, wholesalers, refrigerated transport, platforms, couriers, residential buildings, payment systems, and waste or compost destinations. A PostGIS/Supabase backend can support spatial queries from the MapLibre interface, while D3 can reveal temporal change, route dependence, and disruption.

### Longer-term computational direction
Future versions can compare multiple cost functions: distance, slope, observed travel time, heat exposure, safety, curb access, affordability, and delivery restrictions. Rather than presenting one optimized route as neutral, the interface could allow users to change weights and see how different bodies, workers, and institutions produce different maps of “nearest.”

## 26. Reproducibility and 100-point submission checklist

### Environment and execution
- [ ] The course Conda environment is active.
- [ ] `pip install -r requirements.txt` completes without errors.
- [ ] The notebook runs from top to bottom with no red error cells.
- [ ] Every code cell has an execution number and saved output before submission.

### Data quality
- [ ] MapPLUTO and retail-store record counts are visible.
- [ ] Store filter rule and floor-area coverage are reported.
- [ ] Proportional allocation residual is below 0.5%.
- [ ] Origins and stores beyond the snap tolerance are counted.
- [ ] Corrected network distances are not materially below Euclidean distances.

### Analysis
- [ ] Euclidean and door-to-door network distance are compared.
- [ ] Ten-minute network access includes both snap distances.
- [ ] Centroid and proportional demand allocation are compared.
- [ ] Three priority scenarios, rank correlations, and top-cell overlaps are shown.
- [ ] Three closure scenarios are reported.
- [ ] Extreme cases and representative routes are visually inspected.

### Communication
- [ ] Project diagram is visible near the beginning.
- [ ] Key findings contain actual numbers from the completed run.
- [ ] Principal maps and charts are saved in `figures/`.
- [ ] Diagnostic and summary tables are saved in `tables/`.
- [ ] MapLibre GeoJSON and summary files are generated in `web/data/`.
- [ ] The website is opened through Live Server and all controls work.
- [ ] Bias, limitations, semester reflection, and data sources are included.

Run `python validate_submission.py` after executing the notebook. A fully ready package should report **PASS** for every required artifact and no placeholder findings.

In [ ]:
required_outputs = [
    WEB_DATA_DIR / "access_cells.geojson",
    WEB_DATA_DIR / "stores.geojson",
    WEB_DATA_DIR / "study_core.geojson",
    WEB_DATA_DIR / "study_boundary.geojson",
    WEB_DATA_DIR / "closure_stores.geojson",
    WEB_DATA_DIR / "representative_routes.geojson",
    WEB_DATA_DIR / "summary.json",
    DATA_DIR / "generated_findings.md",
    DATA_DIR / "run_metadata.json",
    TABLE_DIR / "priority_cells.csv",
    TABLE_DIR / "district_summary.csv",
    TABLE_DIR / "closure_scenarios.csv",
]
missing_outputs = [
    str(path.relative_to(PROJECT_ROOT)) for path in required_outputs if not path.exists()
]
figure_count = len(list(FIGURE_DIR.glob("*.png")))

validation_summary = pd.Series(
    {
        "Required output files present": not missing_outputs,
        "Missing output files": ", ".join(missing_outputs) if missing_outputs else "None",
        "Saved figure count": figure_count,
        "Allocation residual below 0.5%": abs(ALLOCATION_RESIDUAL_PCT) <= 0.5,
        "Distance consistency violations": int(access["distance_consistency_flag"].sum()),
        "Priority scenarios": len(PRIORITY_WEIGHT_SCENARIOS),
        "Closure scenarios": len(closure_summary),
    }
)
validation_summary